# **Noise2Void (2D) — powered by CAREamics**

---

<font size = 4> Noise2Void is a deep-learning method that can be used to denoise many types of images, including microscopy images, originally published by [Krull *et al.* on arXiv](https://arxiv.org/abs/1811.10980). It denoises image data in a self-supervised manner, so high-quality, low-noise reference images are **not** required to train the network. A random subset of pixels is "masked" and the network learns to predict their values; the output is a denoised version of the image.

<font size = 4> **This notebook runs Noise2Void on 2D datasets using [CAREamics](https://careamics.github.io/), a modern PyTorch/Lightning implementation. For 3D datasets, use the Noise2Void 3D notebook instead.**

---

<font size = 4>*Disclaimer*:

<font size = 4>This notebook is part of the Zero-Cost Deep-Learning to Enhance Microscopy project (https://github.com/HenriquesLab/DeepLearning_Collab/wiki). Jointly developed by the Jacquemet (https://cellmig.org/) and Henriques (https://henriqueslab.github.io/) laboratories.

<font size = 4>The deep-learning engine used here is **CAREamics** (https://github.com/CAREamics/careamics).

<font size = 4>This notebook is based on:

<font size = 4>**Noise2Void - Learning Denoising from Single Noisy Images**, Krull *et al.*, arXiv 2018 (https://arxiv.org/abs/1811.10980)

<font size = 4>**Please cite the original Noise2Void paper and CAREamics when using this notebook.**

# **How to use this notebook?**

---

<font size = 4>This notebook is structured in numbered sections. Run the cells from top to bottom.

---
### **Structure of a notebook**

<font size = 4>**Text cells** provide information. **Code cells** contain code; move your cursor over the `[ ]` on the left and click the play button to execute.

---
### **Making changes to the notebook**

<font size = 4>**Make a copy** of this notebook and save it to your Google Drive (`File -> Save a copy in Drive`) before editing. To edit a cell, double-click on it.

# **0. Before getting started**
---

<font size = 4>Before running the notebook, make sure you are logged into your Google account and that the training data (and optional quality-control / prediction data) are in your Google Drive.

<font size = 4>Noise2Void requires only noisy images to train — a single noisy image is enough, but multiple images can be used.

<font size = 4>Please note that you can **only use .tif files!**

<font size = 4>**We strongly recommend generating high signal-to-noise versions of some noisy images (a Quality Control dataset).** These paired images let you assess the quality of your trained model directly in this notebook (Section 5).

<font size = 4>A common data structure that works well:

*   Data
    - **Training** — noisy .tif images
    - **Quality control** (optional but recommended)
        - Low SNR images: img_1.tif, img_2.tif ...
        - High SNR images: img_1.tif, img_2.tif ...
    - **Prediction** — images to denoise
    - **Results**

---
<font size = 4>**Important note**

<font size = 4>- To **train from scratch**: run **sections 1–4**, then **section 5** to assess quality and **section 6** to predict.
<font size = 4>- To **evaluate an existing model**: run **sections 1–2**, then **section 5**.
<font size = 4>- To only **run predictions** with an existing model: run **sections 1–2**, then **section 6**.
---

# **1. Install CAREamics and dependencies**
---

## **1.1. Install CAREamics**

In [ ]:
#@markdown ##Install CAREamics and dependencies
#@markdown This installs a pinned, tested version of CAREamics. It may take a minute.

!pip install "careamics==0.3.2" "careamics-portfolio" -q

print("CAREamics installed.")

## **1.2. Restart the runtime (only if prompted)**
<font size = 4>If Colab asks you to restart the runtime after installation, do so (`Runtime -> Restart runtime`), then continue from section 1.3 **without** re-running the install cell.

## **1.3. Load key dependencies**

In [ ]:
#@markdown ##Load key dependencies
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile

import careamics
from careamics import CAREamist
from careamics.config import create_n2v_config
from careamics.utils.metrics import psnr, scale_invariant_psnr

print(f"CAREamics version: {careamics.__version__}")

# **2. Initialise the Colab session**
---

## **2.1. Check for GPU access**

In [ ]:
#@markdown ##Run this cell to check if you have GPU access
import torch

if torch.cuda.is_available():
    print("You have GPU access.")
    print(torch.cuda.get_device_name(0))
else:
    print("You do NOT have GPU access.")
    print("Go to 'Runtime -> Change runtime type' and select a GPU hardware accelerator,")
    print("then re-run the notebook. Expect slow performance on CPU.")

## **2.2. Mount your Google Drive**

In [ ]:
#@markdown ##Play the cell to connect your Google Drive to Colab
#@markdown * Follow the instructions.
#@markdown * Click on "Files" on the left. Refresh it — your Google Drive appears as "drive".

from google.colab import drive
drive.mount('/content/gdrive')

## **2.3. (Optional) Download an example dataset**
<font size = 4>If you just want to try the notebook without your own data, run this cell to download the **SEM** example dataset. It prints a folder path that you can paste into `Training_source` in section 3.1.

In [ ]:
#@markdown ##(Optional) Download the SEM example dataset for testing
Download_example_dataset = True #@param {type:"boolean"}

if Download_example_dataset:
    from careamics_portfolio import PortfolioManager

    portfolio = PortfolioManager()
    files = portfolio.denoising.N2V_SEM.download("./example_data")
    example_folder = str(Path(files[0]).parent)
    print("Example data downloaded to:")
    print(example_folder)
    print("Paste the path above into 'Training_source' in section 3.1.")

# **3. Select your parameters and paths**
---

## **3.1. Setting the main training parameters**
<font size = 4>Set the paths and training parameters below. `Training_source` should point to a folder of noisy `.tif` images (or a single `.tif`).

In [ ]:
#@markdown ###Path to training image(s) (a folder of .tif files, or a single .tif):
Training_source = "" #@param {type:"string"}

#@markdown ### Model name and output folder:
model_name = "my_n2v_model" #@param {type:"string"}
model_path = "" #@param {type:"string"}

#@markdown ###Training parameters
#@markdown Number of epochs:
number_of_epochs = 100 #@param {type:"number"}
#@markdown Patch size (pixels, square):
patch_size = 64 #@param {type:"number"}
#@markdown Batch size:
batch_size = 128 #@param {type:"number"}
#@markdown Number of patches held out for validation:
n_val_patches = 8 #@param {type:"number"}

#@markdown ###Use N2V2 (an improved variant that reduces checkerboard artefacts):
use_n2v2 = False #@param {type:"boolean"}

## **3.2. Data augmentation**
<font size = 4>Data augmentation (flips and 90° rotations) usually improves results and is recommended.

In [ ]:
#@markdown ##Enable or disable data augmentation:
Use_Data_augmentation = True #@param {type:"boolean"}

## **3.3. Using a pre-trained model**
<font size = 4>You can continue training from a previously trained CAREamics model. Provide the path to a checkpoint (`.ckpt`). The pre-trained model's configuration is reused, so the parameters above are ignored when this is enabled.

In [ ]:
#@markdown ##Load weights from a pre-trained CAREamics model
Use_pretrained_model = False #@param {type:"boolean"}
#@markdown ###If enabled, provide the path to the checkpoint (.ckpt) file:
pretrained_model_path = "" #@param {type:"string"}

# **4. Train the network**
---

## **4.1. Prepare the training data and model**

In [ ]:
#@markdown ##Create the configuration and the CAREamist
# Augmentations: None -> default (flips + 90-degree rotations); [] -> disabled
augmentations = None if Use_Data_augmentation else []

work_dir = Path(model_path) if model_path else Path(".")
work_dir.mkdir(parents=True, exist_ok=True)

if Use_pretrained_model:
    print(f"Loading pre-trained model from: {pretrained_model_path}")
    careamist = CAREamist(checkpoint_path=pretrained_model_path, work_dir=work_dir)
else:
    config = create_n2v_config(
        experiment_name=model_name,
        data_type="tiff",
        axes="YX",
        patch_size=(patch_size, patch_size),
        batch_size=batch_size,
        num_epochs=number_of_epochs,
        n_val_patches=n_val_patches,
        use_n2v2=use_n2v2,
        augmentations=augmentations,
    )
    print(config)
    careamist = CAREamist(config, work_dir=work_dir)

## **4.2. Start training**
<font size = 4>Training checkpoints are saved automatically to your output folder. You can monitor the loss below and in Section 5.

In [ ]:
#@markdown ##Start training
careamist.train(train_data=Path(Training_source))
print("Training complete.")

# **5. Evaluate your model**
---

## **5.1. Inspection of the loss function**

In [ ]:
#@markdown ##Plot the training and validation loss vs. epoch
loss_dict = careamist.get_losses()
plt.figure(figsize=(8, 5))
plt.plot(loss_dict["train_epoch"], loss_dict["train_loss"], label="Train loss")
plt.plot(loss_dict["val_epoch"], loss_dict["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training losses")
plt.show()

## **5.2. Quality metrics estimation**
<font size = 4>If you have a Quality Control dataset (paired low-SNR and high-SNR images), the notebook denoises the low-SNR images and compares them against the high-SNR targets using PSNR and scale-invariant PSNR.

In [ ]:
#@markdown ##Provide the Quality Control folders (paired low-SNR and high-SNR .tif images)
Source_QC_folder = "" #@param {type:"string"}
Target_QC_folder = "" #@param {type:"string"}

source_files = sorted(Path(Source_QC_folder).glob("*.tif"))
target_files = sorted(Path(Target_QC_folder).glob("*.tif"))

psnrs, si_psnrs = [], []
for src_f, tgt_f in zip(source_files, target_files):
    gt = tifffile.imread(tgt_f).astype(np.float32)
    pred, _ = careamist.predict(pred_data=str(src_f), tile_size=(256, 256))
    pred_img = np.asarray(pred[0]).squeeze()
    data_range = gt.max() - gt.min()
    psnrs.append(psnr(gt, pred_img, data_range=data_range))
    si_psnrs.append(scale_invariant_psnr(gt, pred_img))
    print(f"{src_f.name}: PSNR={psnrs[-1]:.2f}, SI-PSNR={si_psnrs[-1]:.2f}")

if psnrs:
    print(f"\nMean PSNR:    {np.mean(psnrs):.2f} +/- {np.std(psnrs):.2f}")
    print(f"Mean SI-PSNR: {np.mean(si_psnrs):.2f} +/- {np.std(si_psnrs):.2f}")

# **6. Using the trained model**
---

## **6.1. Generate predictions from an unseen dataset**

In [ ]:
#@markdown ###Path to the data to denoise and the folder where results are saved:
Data_folder = "" #@param {type:"string"}
Result_folder = "" #@param {type:"string"}

result_dir = Path(Result_folder)
result_dir.mkdir(parents=True, exist_ok=True)

predictions, sources = careamist.predict(
    pred_data=Path(Data_folder),
    tile_size=(256, 256),
)

for pred, source in zip(predictions, sources):
    out_name = Path(source).stem + "_denoised.tif"
    out_path = result_dir / out_name
    tifffile.imwrite(out_path, np.asarray(pred).squeeze().astype(np.float32))
    print(f"Saved: {out_path}")

## **6.2. Assess the predicted output**

In [ ]:
#@markdown ##Display an input image next to its denoised prediction
idx = 0 #@param {type:"number"}

input_files = sorted(Path(Data_folder).glob("*.tif"))
input_img = tifffile.imread(input_files[idx])
pred_img = np.asarray(predictions[idx]).squeeze()

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(input_img, cmap="gray")
ax[0].set_title("Input (noisy)")
ax[1].imshow(pred_img, cmap="gray")
ax[1].set_title("Prediction (denoised)")
for a in ax:
    a.axis("off")
plt.show()

# **7. Version log**
---
<font size = 4>**v2.0 (CAREamics)**:
*   First release of the CAREamics-powered Noise2Void 2D notebook, replacing the TensorFlow/CSBDeep implementation.
*   Uses CAREamics 0.3.2 (`create_n2v_config` + `CAREamist`), with optional N2V2.

# **Thank you for using Noise2Void 2D!**